# TaleWeaver — Code Flow Walkthrough

This notebook traces exactly what happens in a session — no UI needed.

## Full Request Flow

```
BEFORE SESSION (StoryScreen mounts)
  POST /api/story-plan   → ADK LlmAgent → 4-beat story arc + opening_text hint      ← NEW (google.adk)
  POST /api/story-opening → Flash Lite (opening text) + Image model (first image)
                           → image held in prewarmImageRef, not shown until "Begin" is clicked

SESSION START (child clicks "Begin")
  seedInitialImage() → opening image appears in canvas for the first time
  WebSocket /ws/story → proxy.py → Gemini Live API (voice, continues from opening)
    ↳ generate_illustration tool → POST /api/image (skip_extraction) → Image model
    ↳ awardBadge tool → BadgePopup in browser
    ↳ movement challenges every ~60s — character asks child to jump/spin/roar

AFTER SESSION ENDS (child taps "📖 See our story!")
  POST /api/story-recap → session images (base64) → Gemini narrates each → interleaved storybook ← NEW
```

**Prerequisites:** Run from the repo root, with `backend/.env` set up.

In [ ]:
import sys, os
sys.path.insert(0, '../backend')

from dotenv import load_dotenv
load_dotenv('../backend/.env')

print('PROJECT:', os.environ.get('GOOGLE_CLOUD_PROJECT', '❌ not set'))
print('LOCATION:', os.environ.get('GOOGLE_CLOUD_LOCATION', 'us-central1'))
print('IMAGE_MODEL:', os.environ.get('IMAGE_MODEL', '❌ not set'))

---
## Step 1 — Characters

Each character has:
- `id` — used by the browser to identify who was chosen
- `name` — shown in the UI
- `voice_name` — which Gemini voice to use
- `image_style` — injected into every image prompt
- `system_prompt` — the full personality and rules for Gemini

In [ ]:
from characters import CHARACTERS, get_character

print(f'{len(CHARACTERS)} characters available:\n')
for char in CHARACTERS.values():
    print(f'  {char.id:15s}  voice={char.voice_name:12s}  {char.name}')

In [ ]:
# Pick one and inspect it
char = get_character('wizard')

print(f'Name:        {char.name}')
print(f'Voice:       {char.voice_name}')
print(f'Image style: {char.image_style}')
print(f'\n--- System prompt (first 600 chars) ---')
print(char.system_prompt[:600], '...')

---
## Step 2 — Gemini Live API Setup Message

When the browser connects via WebSocket, the backend sends this setup message to Gemini Live.

It includes two tools Gemini can call mid-story:
- `generate_illustration` — triggers a storybook image at a vivid moment
- `awardBadge` — awards an achievement badge for genuine engagement (max 2 per session)

Story branching (`showChoice`) was removed — all decisions happen through voice interaction.

In [ ]:
import json
from characters import build_gemini_setup_message

PROJECT_ID = os.environ['GOOGLE_CLOUD_PROJECT']
LOCATION = os.environ.get('GOOGLE_CLOUD_LOCATION', 'us-central1')

setup_msg = build_gemini_setup_message(char, PROJECT_ID, LOCATION)

# Print the structure without the long system prompt
display_msg = json.loads(json.dumps(setup_msg))
display_msg['setup']['system_instruction']['parts'][0]['text'] = '... (system prompt truncated)'

print(json.dumps(display_msg, indent=2))

---
## Step 3 — WebSocket Proxy Flow

After setup, `proxy.py` runs two async tasks in parallel:

```
Browser  ──(audio chunks)──▶  proxy_browser_to_gemini  ──▶  Gemini Live
Browser  ◀──(audio + text)──  proxy_gemini_to_browser  ◀──  Gemini Live
```

The browser sends **raw 16kHz PCM audio** (as base64 JSON). Gemini responds with **24kHz PCM audio** plus text transcripts.

Below is the exact shape of what the browser sends every ~100ms:

In [ ]:
# Shape of audio message from browser → Gemini (the proxy forwards this as-is)
audio_message_shape = {
    'realtime_input': {
        'media_chunks': [
            {
                'mime_type': 'audio/pcm;rate=16000',
                'data': '<base64 encoded 16kHz PCM audio chunk>'
            }
        ]
    }
}

# Shape of response from Gemini → browser
gemini_response_shape = {
    'serverContent': {
        'modelTurn': {
            'parts': [
                {'inlineData': {'mimeType': 'audio/pcm;rate=24000', 'data': '<base64 PCM audio>'}}
            ]
        },
        'outputTranscription': {'text': 'Once upon a time...'},  # what Gemini said
        'inputTranscription': {'text': 'Tell me more!'},          # what the child said
        'turnComplete': True,
    }
}

# Tool call shape (when Gemini decides to award a badge or show choices)
tool_call_shape = {
    'toolCall': {
        'functionCalls': [
            {
                'id': 'call_abc123',
                'name': 'awardBadge',
                'args': {'emoji': '⭐', 'name': 'Story Spark', 'reason': 'Suggested a dragon character'}
            }
        ]
    }
}

print('audio_message  →', json.dumps(audio_message_shape, indent=2)[:200], '...')
print('\ntool_call      →', json.dumps(tool_call_shape, indent=2))

---
## Step 4 (NEW) — Story Pre-warm: `POST /api/story-opening`

**The problem it solves:** the canvas used to be blank for the first 15–20 seconds while Gemini Live warmed up and narrated the opening. Now, a fast pre-warm runs as soon as `StoryScreen` mounts — before the child even clicks "Begin Story!".

**Two-stage pipeline (single Flash Lite call):**
1. `_generate_opening()` prompts Flash Lite to output:
   - `STORY:` — 3-4 sentence opening **in the character's language** (Hindi, Tamil, etc.)
   - `SCENE:` — 1-2 sentence English painter's brief for image gen
2. Image model renders the first illustration from the scene description

**Image timing:** The prewarm image is stored in `prewarmImageRef` in `StoryScreen.tsx` and **not** shown yet — the canvas stays empty. When the child clicks "Begin Story!", `handleBegin()` calls `seedInitialImage()` first, then `connect()`. The opening image appears at the exact moment the story starts.

**Gemini Live continuity — `_begin_turns()`:** `opening_text` is sent in the WebSocket handshake. In `proxy.py`, `_begin_turns()` builds the initial `client_content` message:
1. A `user` turn with "Begin!" + the theme instruction
2. A `model` turn containing `opening_text` (injected as **fake model words**)

Gemini Live sees the model turn as words it already said and continues naturally from that exact story — same characters, same scene — matching the opening image already on screen.

> Requires GCP credentials

In [ ]:
from image_gen import _generate_opening, StoryOpeningRequest, generate_story_opening

# Inspect what _generate_opening produces (text only — no image yet)
wizard = get_character('wizard')
opening_text, scene_description = await _generate_opening(
    character_name=wizard.name,
    character_language='English',
    theme='Space',
    prop_description='',
)

print('=== STORY OPENING ===')
print(opening_text)
print('\n=== SCENE FOR IMAGE GEN ===')
print(scene_description)

In [ ]:
import base64
from IPython.display import Image, display

# Full /api/story-opening — generates opening + first image in one call
dadi = get_character('dadi')

request = StoryOpeningRequest(
    character_id='dadi',
    character_name='Dadi Maa',
    character_language='Hindi',
    image_style=dadi.image_style,
    theme='Animals',
    prop_description='',
)

result = await generate_story_opening(request)

print('Opening text (Hindi):')
print(result.opening_text)
print('\nScene description (English, for image gen):')
print(result.scene_description)
print(f'\nImage: {result.mime_type}, {len(result.image_data)} chars')
display(Image(data=base64.b64decode(result.image_data)))

---
## Step 4b (NEW) — `generate_illustration` Tool Call Flow

During the story, Gemini Live decides *when* to call `generate_illustration` at visually rich moments. The frontend handles it in `useLiveAPI.ts`:

```
Gemini calls generate_illustration({ scene_description: "..." })
  → useLiveAPI immediately sends toolResponse (Gemini continues narrating — NOT blocked)
  → forceImageGeneration(description) called:
      → skip_extraction: true (Gemini's own description is already painter-ready)
      → bypasses rate-limit check (tool images always fire)
      → POST /api/image → image gen → StorySceneGrid
```

The `turnComplete` fallback (`triggerImageGeneration`) still runs at the user-configured interval for dialogue-heavy turns where Gemini doesn't call the tool.

In [ ]:
# Demonstrate skip_extraction — Gemini writes the scene description, no Flash Lite needed
from image_gen import ImageRequest, generate_scene_image

# Tool-triggered: scene_description is Gemini's own painter-ready text
tool_request = ImageRequest(
    scene_description="Wizard Wally standing on a silver moon crater, arms outstretched, conjuring a spiral of glowing blue star-sparks into the vast starry void above him, his purple robes billowing in zero gravity.",
    story_context="",
    image_style=wizard.image_style,
    session_id='notebook-tool-test',
    skip_extraction=True,         # ← key: bypass Flash Lite, use description as-is
    previous_image_data='',
    previous_image_mime_type='image/png',
    previous_scene_description='',
)

tool_result = await generate_scene_image(tool_request)
print(f'Scene used directly: {tool_result.scene_description}')
display(Image(data=base64.b64decode(tool_result.image_data)))

---
## Step 4 — Theme Safety Check

Before a session starts, the browser calls `POST /api/check-theme` to validate custom themes. It uses **Gemini 2.0 Flash Lite** to do a quick SAFE/UNSAFE classification.

> Requires GCP credentials

In [ ]:
import asyncio
from image_gen import _is_safe_for_children

themes_to_test = [
    'dragons and treasure',
    'pirates on the ocean',
    'fighting with swords',
    'birthday party animals',
]

for theme in themes_to_test:
    safe = await _is_safe_for_children(theme)
    status = '✅ SAFE' if safe else '❌ UNSAFE'
    print(f'{status}  →  {theme}')

---
## Step 5 — Image Generation Pipeline

The browser sends `POST /api/image` with:
- `scene_description` — what Gemini just narrated (in any language!)
- `story_context` — the last few minutes of story transcript
- `image_style` — from the character config
- `previous_image_data` — the last illustration (for visual continuity)

The backend does this in two steps:

```
Step A: Gemini Flash Lite extracts a specific English scene description
Step B: Imagen (or Gemini image model) renders the illustration
```

### Step 5A — Scene Extraction

> Converts messy story narration → precise painter's brief

In [ ]:
from image_gen import _extract_english_scene

# Simulating a Hindi story narration from Dadi Maa
narration = """
एक बार की बात है, एक छोटी सी गौरैया थी जिसका नाम चिन्नी था।
वह एक बड़े बरगद के पेड़ पर रहती थी। एक दिन, चिन्नी को पता चला
कि उसके घोंसले में एक जादुई बीज है!
"""

story_context = "Story about a sparrow named Chinni who lives in a banyan tree"

scene = await _extract_english_scene(narration, story_context)
print('Extracted scene:')
print(scene)

### Step 5B — Build the Image Prompt

The extracted scene is prefixed with a safety prefix and the character's art style.

In [ ]:
from image_gen import SAFETY_PREFIX

dadi = get_character('dadi')

# This is the exact prompt sent to the image model
example_scene = "A tiny brown sparrow named Chinni sitting inside a cozy nest in a giant banyan tree, discovering a glowing golden seed, her eyes wide with wonder, dappled sunlight filtering through the leaves."
full_prompt = f"{SAFETY_PREFIX}{dadi.image_style}, {example_scene}"

print('Safety prefix:')
print(f'  {SAFETY_PREFIX}\n')
print('Image style:')
print(f'  {dadi.image_style}\n')
print('Final prompt sent to image model:')
print(f'  {full_prompt}')

### Step 5C — Generate the Image

> Requires GCP credentials + `IMAGE_MODEL` and `IMAGE_LOCATION` set in `.env`

In [ ]:
import base64
from IPython.display import Image, display
from image_gen import IMAGE_MODEL, _generate_imagen, _generate_gemini, _api_key_client, _generate_gemini_api_key

print(f'Image model: {IMAGE_MODEL}')

if IMAGE_MODEL.startswith('imagen-'):
    image_b64, mime_type = await _generate_imagen(full_prompt)
elif _api_key_client:
    image_b64, mime_type = await _generate_gemini_api_key(full_prompt)
else:
    image_b64, mime_type = await _generate_gemini(full_prompt)

print(f'Generated! mime={mime_type}, size={len(image_b64)} chars (base64)')
display(Image(data=base64.b64decode(image_b64)))

---
## Step 6 — Visual Continuity (Multi-Image Sessions)

For the 2nd+ image in a session, the previous image is passed back in as a reference. The model is told to:
- Keep the same characters if the story is continuing
- Use a fresh background if the scene shifted
- Ignore the reference if it's a completely new cast

In [ ]:
from image_gen import _build_contents

# Without previous image — returns just the prompt string
contents_first = _build_contents(full_prompt, previous_image_data='', previous_image_mime_type='image/png')
print('First image → contents type:', type(contents_first))
print('Value:', contents_first[:80], '...' if len(contents_first) > 80 else '')

# With previous image — returns a list of Parts (instruction + image + new prompt)
fake_prev = base64.b64encode(b'fake_image_bytes').decode()  # placeholder
contents_subsequent = _build_contents(full_prompt, previous_image_data=fake_prev, previous_image_mime_type='image/png', previous_scene_description='Chinni the sparrow in her nest')
print('\nSubsequent image → contents type:', type(contents_subsequent))
print('Number of parts:', len(contents_subsequent))
for i, part in enumerate(contents_subsequent):
    if hasattr(part, 'text') and part.text:
        print(f'  Part {i} (text): {part.text[:120]}...')
    elif hasattr(part, 'inline_data'):
        print(f'  Part {i} (image): {part.inline_data.mime_type}')

---
## Step 7 — Full `/api/image` Endpoint

This is what the browser actually calls. It wraps the two steps above.

In [ ]:
from image_gen import ImageRequest, generate_scene_image

request = ImageRequest(
    scene_description="एक छोटी सी गौरैया थी जिसका नाम चिन्नी था।",  # Hindi narration
    story_context="Sparrow named Chinni in a banyan tree finds a magic seed",
    image_style=dadi.image_style,
    session_id='notebook-test-001',
    previous_image_data='',
    previous_image_mime_type='image/png',
    previous_scene_description='',
)

response = await generate_scene_image(request)

print(f'Extracted scene: {response.scene_description}')
print(f'Image: {response.mime_type}, {len(response.image_data)} chars')
display(Image(data=base64.b64decode(response.image_data)))

---
## Step 8 — Story Planner: ADK `LlmAgent`  ← NEW

Before a session starts, the **Story Planner** generates a structured story outline using Google ADK. This is TaleWeaver's explicit ADK layer — a `google.adk.agents.LlmAgent` backed by `gemini-2.0-flash`.

**File:** `backend/story_planner.py` · **Endpoint:** `POST /api/story-plan`

### How it works

```
StoryScreen mounts
        │
        ▼
  POST /api/story-plan  { character_id, theme }
        │
        ▼
  google.adk.agents.LlmAgent  (story_planner)
    model: gemini-2.0-flash
    InMemorySessionService  ←  one session per request (plan-{character}-{hex})
    Runner.run_async()      ←  async generator, collect final response event
        │
        ▼
  StoryPlan {
    setting:  "one sentence — where the story takes place"
    hero:     "child protagonist's opening situation"
    beats:    ["3–4 short plot beat strings, ≤10 words each"]
    ending:   "warm resolution sentence"
  }
        │
        ▼
  opening_text hint  "[Story plan] Setting: ... | Start: ... | Moments: ... | Resolution: ..."
        │
        ▼
  Passed as opening_text in the WebSocket init message
  → proxy.py appends it to the "Begin!" trigger
  → Gemini Live character has a creative roadmap from the first word
```

### ADK components used

| Component | Import | Role |
|---|---|---|
| `LlmAgent` | `google.adk.agents` | Wraps a Gemini model with a structured instruction |
| `Runner` | `google.adk.runners` | Executes the agent against a session |
| `InMemorySessionService` | `google.adk.sessions` | In-process session state (no DB needed) |
| `genai_types.Content` | `google.genai.types` | Structures the user message for the runner |

> Requires GCP credentials

In [ ]:
from story_planner import generate_story_plan, StoryPlanRequest

# The ADK LlmAgent generates a 4-beat story arc before the session starts.
# The returned opening_text can be passed in the WebSocket init message so the
# character has a creative roadmap from the very first word.

plan_request = StoryPlanRequest(
    character_id='wizard',
    theme='space adventure',
)

plan_result = await generate_story_plan(plan_request)

print('=== STORY PLAN (from ADK LlmAgent) ===')
print()
print(f'Setting:  {plan_result.plan.setting}')
print(f'Hero:     {plan_result.plan.hero}')
print()
print('Plot beats:')
for i, beat in enumerate(plan_result.plan.beats, 1):
    print(f'  {i}. {beat}')
print()
print(f'Ending:   {plan_result.plan.ending}')
print()
print('=== OPENING TEXT (injected into WebSocket init) ===')
print(plan_result.opening_text)

---
## Step 9 — Story Recap: Session Images + Gemini Narration  ← NEW

After a story ends, the child taps **"📖 See our story!"** to see a full illustrated recap. Rather than generating new images (which produced hallucinated content unrelated to the actual session), the recap uses the **actual session images** as ground truth.

**Files:** `backend/image_gen.py` (endpoint) · `frontend/src/components/StoryRecapModal.tsx` (UI)

### How it works

```
Session images (base64 PNGs from generate_illustration tool calls)
        │  stored in useStoryImages.ts, passed to StoryRecapModal
        ▼
  StoryRecapModal POSTs { character_name, image_style, scenes: [{ image_data, mime_type, description }] }
        │
        ▼
  backend: _narrate_scene() called for each image IN PARALLEL (asyncio.gather)
    │  gemini-2.0-flash receives the actual session image
    │  prompt: "You are {character}. Write 2 warm sentences narrating THIS specific moment."
    │  TEXT-only response (response_modalities=["TEXT"])
    │  20s timeout per image, falls back to scene.description on error
        │
        ▼
  pages: text narration + original session image, interleaved per scene
        │
        ▼
  StoryRecapModal renders alternating <p> and <img>
  — narrations describe what actually happened, images are exactly what the child saw
```

**Why this approach:** The old method sent short text descriptions to Gemini and asked it to generate new pages with new images — it produced entirely hallucinated content with no connection to the actual story. Using the real session images as input guarantees the recap reflects what actually happened. Parallel narration is also much faster (~20s for all scenes vs ~90s for sequential image generation).

> Requires GCP credentials

In [ ]:
import base64
from IPython.display import Image, display
from image_gen import generate_story_recap, StoryRecapRequest, RecapScene

# In a real session, scenes come from generate_illustration tool calls tracked in
# useStoryImages.ts. The frontend sends the actual base64 image data so the recap
# faithfully reflects what was shown during the story.
#
# Here we reuse the image we generated in Step 7 as a mock session image.
mock_image_b64 = response.image_data   # from Step 7: Chinni the sparrow scene
mock_image_mime = response.mime_type

mock_scenes = [
    RecapScene(
        image_data=mock_image_b64,
        mime_type=mock_image_mime,
        description="Chinni the sparrow discovers a magic seed",
    ),
    RecapScene(
        image_data=mock_image_b64,   # reusing same image for demo
        mime_type=mock_image_mime,
        description="Chinni plants the seed and a giant flower blooms",
    ),
]

recap_request = StoryRecapRequest(
    character_name='Dadi Maa',
    image_style=dadi.image_style,
    scenes=mock_scenes,
)

result = await generate_story_recap(recap_request)

print(f'Generated {len(result.pages)} storybook pages:')
print()
for i, page in enumerate(result.pages):
    if page.type == 'text':
        print(f'--- Narration ---')
        print(page.content)
        print()
    else:
        print(f'--- Illustration (original session image, {page.mime_type}) ---')
        display(Image(data=base64.b64decode(page.image_data)))

---
## Summary — Full Data Flow Map

```
═══════════════════════════════════════════════════════════════
  BEFORE SESSION  (StoryScreen mounts, parallel requests)
═══════════════════════════════════════════════════════════════

BROWSER  POST /api/story-plan  { character_id, theme }           ← ADK (google.adk)
  │
  ▼
FastAPI  /api/story-plan  →  story_planner.py
  ├─ LlmAgent(model=gemini-2.0-flash)  +  Runner  +  InMemorySessionService
  ├─ Runner.run_async() → final response event → parse JSON
  └─ returns StoryPlan { setting, hero, beats[], ending }
           + opening_text hint for the WebSocket init message

BROWSER  POST /api/story-opening  { character_id, theme, prop_description? }
  │
  ▼
FastAPI  /api/story-opening  →  image_gen.py
  ├─ _generate_opening()           Model: gemini-2.0-flash-lite
  │    → STORY: opening in character's language (Hindi, Tamil, etc.)
  │    → SCENE: English painter's brief for image gen
  ├─ image generation              Model: IMAGE_MODEL env var
  └─ returns { opening_text, image_data, scene_description }
       ↓
  Browser stores image in prewarmImageRef — canvas stays empty until "Begin" is clicked

═══════════════════════════════════════════════════════════════
  SESSION  (child clicks "Begin Story!")
═══════════════════════════════════════════════════════════════

handleBegin() → seedInitialImage(prewarmImageRef.current)  ← opening image appears NOW
             → connect()

BROWSER  WS connect + { character_id, theme, prop_image?, opening_text }
  │
  ▼
FastAPI  /ws/story  →  proxy.py
  │
  ├─ get_character(id)              → Character(name, voice, style, system_prompt)
  ├─ build_gemini_setup_message()   → JSON setup (tools: generate_illustration, awardBadge)
  ├─ WS connect to Gemini Live API  → authenticated via GCP ADC
  ├─ _begin_turns(theme, prop_image, opening_text):
  │    turn 1 (user):  "Begin! The theme is: {theme}."
  │    turn 2 (model): opening_text  ← fake model words → Gemini continues from here
  │    → Gemini Live treats opening_text as its own prior words, matching the shown image
  │
  │  [BIDIRECTIONAL PROXY LOOP]
  │  Browser 16kHz PCM audio → Gemini Live
  │  Gemini 24kHz PCM audio + transcripts → Browser
  │
  │  System prompt: movement challenge every ~60s
  │    → character instructs child to jump/spin/roar, pauses 15–20s for response
  │    → camera stream (if enabled) lets Gemini react to movement with delight
  │
  │  Gemini tool call: generate_illustration({ scene_description })
  │    → useLiveAPI sends toolResponse immediately (narration unblocked)
  │    → forceImageGeneration(description, skip_extraction=true)
  │         → POST /api/image → image gen → StorySceneGrid shimmer → fade-in
  │
  │  Gemini tool call: awardBadge({ emoji, name, reason })
  │    → BadgePopup shown in browser
  │
  │  turnComplete fallback → triggerImageGeneration (rate-limited, dialogue turns)
  │
FastAPI  /api/image  { scene_description, story_context, image_style, prev_image?, skip_extraction }
  │
  ├─ if skip_extraction=False:
  │    _extract_english_scene()     Model: gemini-2.0-flash-lite
  │    Any-language transcript → specific English painter's brief
  │
  └─ image generation               Model: IMAGE_MODEL env var
       _build_contents(): text-only (first) or [instruction + prev_image + prompt] (2nd+)
       → base64 PNG returned to browser → StorySceneGrid shimmer → fade-in

═══════════════════════════════════════════════════════════════
  AFTER SESSION  (child taps "📖 See our story!")
═══════════════════════════════════════════════════════════════

BROWSER  POST /api/story-recap  { character_name, image_style, scenes: [{ image_data, mime_type }] }
  │  (actual session images from useStoryImages.ts — not text descriptions)
  ▼
FastAPI  /api/story-recap  →  image_gen.py
  ├─ _narrate_scene() called for each image IN PARALLEL (asyncio.gather)
  │    gemini-2.0-flash receives actual session image + character prompt
  │    → 2 warm sentences narrating what's happening in that specific image
  │    → TEXT-only (response_modalities=["TEXT"]), 20s timeout per scene
  │
  └─ returns pages: [{ type:"text", narration }, { type:"image", original_session_image }]
       ↓
  StoryRecapModal renders alternating <p> and <img>
  — narrations faithfully describe what actually happened, images are exactly what the child saw
```

---

### Models Used

| Step | Endpoint | Purpose | Model |
|---|---|---|---|
| Pre-session | `POST /api/story-plan` | 4-beat story arc (ADK LlmAgent) | `gemini-2.0-flash` |
| Pre-session | `POST /api/story-opening` | Opening text + scene brief | `gemini-2.0-flash-lite` |
| Pre-session | `POST /api/story-opening` | First illustration | `IMAGE_MODEL` env var |
| Session | `WS /ws/story` | Voice conversation | `gemini-live-2.5-flash-native-audio` |
| Session | `POST /api/image` (fallback) | Scene extraction from transcript | `gemini-2.0-flash-lite` |
| Session | `POST /api/image` (tool) | Scene used as-is (`skip_extraction=True`) | — |
| Session | `POST /api/image` | Illustration generation | `IMAGE_MODEL` env var |
| Utility | `POST /api/check-theme` | Safety classification | `gemini-2.0-flash-lite` |
| Utility | `POST /api/sketch-preview` | Sketch labelling + image | `gemini-2.0-flash-lite` + `IMAGE_MODEL` |
| Post-session | `POST /api/story-recap` | Narrate each session image in parallel | `gemini-2.0-flash` |

### ADK Components

| File | ADK Class | Role |
|---|---|---|
| `backend/story_planner.py` | `LlmAgent`, `Runner`, `InMemorySessionService` | Pre-session story planning |
| `backend/requirements.txt` | `google-adk>=1.0.0` | ADK dependency |

### Key Design Decisions (this session)

| Change | Reason |
|---|---|
| `showChoice` tool removed entirely | Appeared randomly, disappeared randomly — voice handles all decisions |
| `_begin_turns()` injects `opening_text` as a **model turn** | Gemini ignores user-injected instructions; treating it as its own prior words makes it continue naturally |
| Opening image held in `prewarmImageRef` until "Begin" | Image was showing on the pre-Begin screen — now appears exactly when the story starts |
| Recap uses actual session images | Old approach sent 100-char text snippets → Gemini hallucinated new unrelated content |
| Movement challenge every ~60s (was once per story) | Children need more frequent physical engagement |